# Tutorial 01 — Earth-Moon L1 Lyapunov family

Build 30 periodic Lyapunov orbits at L1 in the Earth-Moon CR3BP via amplitude continuation,
then plot them colored by Jacobi constant. The L1/L2 Lyapunov families are the lowest-energy
periodic orbits around the libration points; their stable/unstable manifold tubes are the
natural transport highways of low-energy mission design.

_~3 seconds to run._

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import ariadne
from ariadne.data.constants import EARTH_MOON
from ariadne.dynamics.cr3bp import propagate

MU = EARTH_MOON.mu
L_STAR = EARTH_MOON.L_star
print(f'Earth-Moon: mu = {MU:.6f}, L* = {L_STAR:.0f} km')

In [ ]:
# Build the L1 Lyapunov family by amplitude continuation
family = ariadne.lyapunov_family('L1', n=30)
print(f'L1 Lyapunov family: {len(family)} orbits')
print(f'Jacobi range: {family[0].orbit.jacobi:.3f} -> {family[-1].orbit.jacobi:.3f}')

In [ ]:
# Plot every other orbit, colored by Jacobi constant
fig, ax = plt.subplots(figsize=(9, 7))
cmap = plt.cm.viridis
norm = plt.Normalize(family[-1].orbit.jacobi, family[0].orbit.jacobi)
for m in family[::2]:
    sol = propagate(m.orbit.s0, (0.0, m.orbit.period), MU,
                    t_eval=np.linspace(0.0, m.orbit.period, 200))
    ax.plot(sol.y[0] * L_STAR, sol.y[1] * L_STAR, color=cmap(norm(m.orbit.jacobi)), lw=0.7)
ax.scatter([(1 - MU) * L_STAR], [0], s=80, c='gray', label='Moon')
ax.scatter([-MU * L_STAR], [0], s=180, c='royalblue', label='Earth')
ax.set_xlabel('x [km]'); ax.set_ylabel('y [km]')
ax.set_title(f'Earth-Moon L1 Lyapunov family ({len(family)} orbits, amplitude-continued)')
ax.set_aspect('equal'); ax.legend(); ax.grid(alpha=0.3)
plt.colorbar(plt.cm.ScalarMappable(norm, cmap), ax=ax, label='Jacobi constant')
plt.tight_layout()
plt.show()

## What just happened

`ariadne.lyapunov_family` walked the L1 planar Lyapunov family from amplitude 1e-3 up by
30 steps. Each orbit is differentially-corrected so its half-period residual is < 1e-12 —
true periodicity, not just a near-miss. The family's Jacobi constant decreases as amplitude
grows; the manifolds of higher-amplitude orbits reach further out (toward Earth on one side,
toward the Moon on the other), which is what makes them useful for transfer design.